# Phase 1: Full Data Collection Pipeline

Complete end-to-end Phase 1 data collection on Colab with GPU acceleration.

**What this does:**
- Loads Qwen 1.5 MoE model with GPU
- Tokenizes 9.5k prompts (GSM8K + MBPP)
- Captures routing data via forward hooks
- Runs 5 deterministic runs with fixed seeds
- Validates data quality and determinism
- Saves results to Google Drive

**Runtime:** ~3-4 hours on A100 GPU

**Output:** Parquet files ready for Phase 2 analysis

## Setup: Clone Repo & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (use integration branch which has all Phase 1 code)
!cd /content && git clone --branch integration/phase-1-validation https://github.com/scerruti/moe-expert-prefetching.git
%cd /content/moe-expert-prefetching/phase_1/scripts

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets pyarrow tqdm accelerate bitsandbytes -U
print("✅ Dependencies installed")

In [ ]:
# Verify GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Run Phase 1 Collection

This will:
1. Load the model (first time: ~10 min, subsequent: ~2 min)
2. Load datasets (GSM8K + MBPP)
3. Run 5 deterministic collection runs
4. Validate routing data
5. Save parquet files to Google Drive

**Total time: ~3-4 hours**

In [ ]:
import sys
from pathlib import Path
from full_pipeline import run_full_pipeline

# Set output to Google Drive
output_dir = Path("/content/drive/My Drive/moe_phase1_data")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Available space: {output_dir.stat().st_size / 1e9:.1f} GB")

In [ ]:
# Run full Phase 1 collection
print("Starting Phase 1 data collection...")
print("This will take 3-4 hours. Monitor progress below.\n")

stats = run_full_pipeline(
    model_name="qwen1.5-moe-a2.7b",
    num_runs=5,              # Full 5 runs with different seeds
    subset_size=None,        # Use ALL prompts (~9.5k)
    output_dir=output_dir,
)

print("\n" + "="*70)
print("PHASE 1 COLLECTION COMPLETE")
print("="*70)
print(f"\nResults:")
print(f"  Prompts processed: {stats.total_prompts_processed:,}")
print(f"  Routing events captured: {stats.total_routing_events:,}")
print(f"  Runs completed: {stats.runs_completed}/{stats.runs_completed}")
print(f"  Validation passed: {stats.validation_passed}")
print(f"  Errors: {len(stats.errors)}")
print(f"\nOutput saved to: {output_dir}")

## Validate Results

In [ ]:
# List output files
import os
files = list(output_dir.glob("*.parquet"))
print(f"Parquet files created: {len(files)}")
for f in sorted(files):
    size_mb = f.stat().st_size / 1e6
    print(f"  - {f.name}: {size_mb:.1f} MB")

In [ ]:
# Run determinism validation on collected data
from determinism_validation import DeterminismValidator

print("Running determinism validation on collected data...\n")
validator = DeterminismValidator(output_dir)
report = validator.validate_all()

print("\n" + report.summary)

## Download & Archive

Data is saved to Google Drive. You can now:
1. Download the parquet files
2. Use them for Phase 2 analysis
3. Share with team

In [ ]:
# Show download path
print(f"✅ Phase 1 data collection complete!")
print(f"\nYour data is in Google Drive:")
print(f"  My Drive → moe_phase1_data/")
print(f"\nReady for Phase 2 analysis!")